# RCM360 source EDA

This notebook profiles the pinned Misata synthetic dataset and summarizes claims, submission cohorts, and encounter context. It reads the verified local archive through the project parser, leaves source files unchanged, and does not display patient or provider identifiers or row-level records.

All results are descriptive of this synthetic snapshot. PostgreSQL remains the required execution environment for final SQL findings and financial reconciliation; this notebook is exploratory and does not replace those checks.

In [ ]:
from decimal import Decimal
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "etl" / "source.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the RCM360 project tree.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.etl.source import SOURCE_SHA256, TABLE_COLUMNS, read_snapshot

RAW_DIR = ROOT / "data" / "raw"
sns.set_theme(style="whitegrid", palette=["#5145B8", "#8784B5", "#DBA14A", "#167451"])
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_rows", 30)
pd.set_option("display.max_columns", 20)

snapshot = read_snapshot(RAW_DIR)
frames = {
    table: pd.DataFrame.from_records(rows, columns=TABLE_COLUMNS[table])
    for table, rows in snapshot.items()
}
row_counts = pd.Series({table: len(frame) for table, frame in frames.items()}, name="rows")
assert int(row_counts.sum()) == 39_946
print(f"Verified archive SHA-256: {SOURCE_SHA256}")
display(row_counts.to_frame())

## Source structure and completeness

The next tables expose row counts, column counts, key uniqueness, and missing-cell totals only. They do not print source records or identifier values.

In [ ]:
primary_keys = {
    "patients": "patient_id", "providers": "provider_id",
    "encounters": "encounter_id", "diagnoses": "diagnosis_id",
    "prescriptions": "prescription_id", "lab_results": "lab_id",
    "claims": "claim_id",
}
structure = pd.DataFrame([
    {
        "table": table, "rows": len(frame), "columns": len(frame.columns),
        "duplicate_primary_keys": int(frame[primary_keys[table]].duplicated().sum()),
        "columns_with_missing_values": int(frame.isna().any().sum()),
        "missing_cells": int(frame.isna().sum().sum()),
    }
    for table, frame in frames.items()
])
assert structure["duplicate_primary_keys"].sum() == 0
display(structure)

missingness = (
    pd.concat(
        [frame.isna().sum().rename(table).to_frame() for table, frame in frames.items()],
        axis=1,
    )
    .rename_axis("field")
    .stack()
    .rename("missing_cells")
    .reset_index()
    .query("missing_cells > 0")
    .sort_values(["level_1", "missing_cells"], ascending=[True, False])
    .rename(columns={"level_1": "table"})
)
display(missingness)

## Claim status and financial fields

Financial totals use the source decimal values directly. paid_amount is the publisher?s recorded payer-payment field; patient responsibility is assigned responsibility, not evidence of receipt. Amounts have no currency field. Denied billed amounts are exposure in the source snapshot, not confirmed losses or recoverable revenue.

In [ ]:
claims = frames["claims"].copy()
amount_fields = (
    "billed_amount", "allowed_amount", "paid_amount",
    "patient_responsibility",
)
status_order = ["paid", "denied", "pending"]
finance_rows = []
for status in status_order:
    subset = claims.loc[claims["status"] == status]
    finance_rows.append({
        "status": status,
        "claim_count": len(subset),
        **{field: sum(subset[field], Decimal(0)) for field in amount_fields},
    })
financial_summary = pd.DataFrame(finance_rows)
assert int(financial_summary["claim_count"].sum()) == len(claims)
display(financial_summary)

status_counts = claims["status"].value_counts().reindex(status_order, fill_value=0)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=status_counts.index, y=status_counts.values, color="#5145B8", ax=ax)
ax.set(title="Claims by observed source status", xlabel="Source status", ylabel="Claims")
for bar, value in zip(ax.patches, status_counts.values, strict=True):
    ax.annotate(f"{value:,}", (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha="center", va="bottom", xytext=(0, 4), textcoords="offset points")
plt.tight_layout()
plt.show()

## Submission-month cohorts

Cohorts use submitted_at. The denial share is denied claims divided by all claims submitted in that month, with volume shown alongside it. The source has no outcome observation cutoff or maturity window, so recent cohorts may be less complete; month-level differences are descriptive and do not establish process change.

In [ ]:
claims["submission_month"] = claims["submitted_at"].dt.to_period("M").dt.to_timestamp()
monthly = claims.assign(is_denied=claims["status"].eq("denied")).groupby("submission_month").agg(
    claim_count=("claim_id", "size"), denied_count=("is_denied", "sum")
)
month_index = pd.date_range(monthly.index.min(), monthly.index.max(), freq="MS")
monthly = monthly.reindex(month_index, fill_value=0).rename_axis("submission_month")
monthly["observed_denial_rate"] = np.divide(
    monthly["denied_count"], monthly["claim_count"],
    out=np.full(len(monthly), np.nan, dtype=float),
    where=monthly["claim_count"].to_numpy() > 0,
)
display(monthly.reset_index().tail(6))

fig, count_ax = plt.subplots(figsize=(11, 5))
count_ax.bar(monthly.index, monthly["claim_count"], color="#8784B5", width=20, label="Claims submitted")
count_ax.set_ylabel("Claims submitted")
count_ax.set_xlabel("Submission month")
rate_ax = count_ax.twinx()
rate_ax.plot(monthly.index, monthly["observed_denial_rate"] * 100, color="#B74660",
             marker="o", linewidth=1.8, label="Observed denial share")
rate_ax.set_ylabel("Observed denial share (%)")
rate_ax.set_ylim(bottom=0)
count_ax.set_title("Submission-month volume and observed status share")
count_ax.tick_params(axis="x", rotation=45)
handles_a, labels_a = count_ax.get_legend_handles_labels()
handles_b, labels_b = rate_ax.get_legend_handles_labels()
count_ax.legend(handles_a + handles_b, labels_a + labels_b, loc="upper left")
fig.tight_layout()
plt.show()

## Payer mix and denial reasons

Payer groups retain the source labels. Denial rates use all claims in each payer group as their denominator. Reason counts are restricted to denied claims and sum to the observed denied subset; a reason label is not independently verified as a root cause.

In [ ]:
payer_summary = claims.assign(is_denied=claims["status"].eq("denied")).groupby("payer", as_index=False).agg(
    claim_count=("claim_id", "size"), denied_count=("is_denied", "sum")
)
payer_summary["observed_denial_rate"] = payer_summary["denied_count"] / payer_summary["claim_count"]
payer_summary = payer_summary.sort_values("claim_count", ascending=False)
display(payer_summary)

denial_reasons = (
    claims.loc[claims["status"] == "denied", "denial_reason"]
    .value_counts(dropna=False).rename_axis("source_reason").rename("claim_count").reset_index()
)
denial_reasons["share_of_denied_claims"] = denial_reasons["claim_count"] / int((claims["status"] == "denied").sum())
display(denial_reasons)
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=denial_reasons, x="claim_count", y="source_reason", color="#5145B8", ax=ax)
ax.set(title="Observed denial-reason labels", xlabel="Denied claims", ylabel="Source label")
plt.tight_layout()
plt.show()

## Encounter context without multiplying claims

Diagnoses, labs, and prescriptions are one-to-many tables. The cell below reduces each to encounter-level counts before any combination with the claim table. The join is validated as one claim per encounter; the summaries describe data availability and do not imply that additional services caused billing outcomes.

In [ ]:
encounters = frames["encounters"]
context = pd.DataFrame({
    "encounter_id": encounters["encounter_id"],
    "diagnosis_rows": encounters["encounter_id"].map(frames["diagnoses"].groupby("encounter_id").size()).fillna(0).astype(int),
    "lab_rows": encounters["encounter_id"].map(frames["lab_results"].groupby("encounter_id").size()).fillna(0).astype(int),
    "prescription_rows": encounters["encounter_id"].map(frames["prescriptions"].groupby("encounter_id").size()).fillna(0).astype(int),
})
assert len(context) == len(encounters)
assert context["diagnosis_rows"].equals(encounters["diagnosis_count"])
claims_with_context = claims.merge(
    context, on="encounter_id", how="left", validate="one_to_one"
)
assert len(claims_with_context) == len(claims)
context_summary = pd.DataFrame([
    {
        "related_table": table,
        "rows": len(frames[table]),
        "encounters_with_rows": int(context[count_field].gt(0).sum()),
        "median_rows_per_encounter": float(np.median(context[count_field].to_numpy())),
    }
    for table, count_field in (("diagnoses", "diagnosis_rows"),
                               ("lab_results", "lab_rows"),
                               ("prescriptions", "prescription_rows"))
])
display(context_summary)

## Encounter duration and provider concentration

Length-of-stay summaries are grouped by encounter type. Provider concentration is computed after the verified one-to-one claim-to-encounter join, but only rank and volume are displayed; identifiers are not exposed. These are sample descriptions, not measures of quality, efficiency, or real-world provider performance.

In [ ]:
encounters["los_hours_numeric"] = pd.to_numeric(encounters["los_hours"], errors="raise")
los_summary = encounters.groupby("encounter_type")["los_hours_numeric"].agg(
    encounter_count="size", median_hours="median",
    q1_hours=lambda values: values.quantile(0.25),
    q3_hours=lambda values: values.quantile(0.75),
).reset_index()
display(los_summary)
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=encounters, x="encounter_type", y="los_hours_numeric",
            color="#C7C3E5", showfliers=False, ax=ax)
ax.set(title="Source length-of-stay distribution", xlabel="Encounter type", ylabel="Hours")
plt.tight_layout()
plt.show()

provider_encounters = encounters[["encounter_id", "provider_id"]]
provider_claims = claims[["encounter_id", "claim_id"]].merge(
    provider_encounters, on="encounter_id", validate="one_to_one"
)
provider_volume = provider_claims.groupby("provider_id").size().sort_values(ascending=False)
assert int(provider_volume.sum()) == len(claims)
provider_shares = provider_volume.to_numpy(dtype=float) / len(claims)
concentration = pd.DataFrame({
    "provider_rank_by_claim_volume": np.arange(1, len(provider_shares) + 1),
    "claim_count": provider_volume.to_numpy(),
    "cumulative_claim_share": np.cumsum(provider_shares),
})
print(f"Top 12 provider share of claims: {concentration.head(12)['claim_count'].sum() / len(claims):.1%}")
display(concentration.head(12))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(concentration["provider_rank_by_claim_volume"],
        concentration["cumulative_claim_share"] * 100, color="#5145B8")
ax.set(title="Cumulative claims by provider rank", xlabel="Provider rank (volume only)",
       ylabel="Cumulative share of claims (%)")
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

## Interpretation and next verification

- Treat every plot and table as a description of the approved synthetic snapshot, not evidence about real healthcare operations.
- Submission-month rates have no known maturity window; compare them with counts and avoid reading recent differences as performance changes.
- Payer payment, assigned patient responsibility, billed value, and allowed value remain distinct source fields. No collection rate, aging, cash realization, or denial recovery is available.
- One-to-many clinical context was pre-aggregated before claim joins, and the resulting claim count was asserted unchanged.
- The SQL analysis pack, QA audit, and JSON exporter must still be run and reconciled on PostgreSQL before their outputs are presented as final project findings.
- Notebook outputs should be cleared before committing. The raw archive, CSVs, row-level identifiers, and local audit artifacts remain outside the public repository.